In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

# 6.01 — Within-subject medication effect on shared-basis state occupancy

All 41 PD subjects have both OFF and ON sessions. On the **shared HC+PD basis** the states
mean the same thing for every session, so the medication effect is a **within-subject**
change in state occupancy. Primary readout: occupancy of the **color-bias state** (the
state with the largest `color` weight = strongest prior integration).

Hypothesis: color-bias-state occupancy rises OFF→ON within tremor subjects, stays flat in
brady, with HC as a high reference. Requires `all_subjects_final.pkl` from
`finetune_shared_basis.py`.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import statsmodels.formula.api as smf

from imports import *
from config import dir_config
from src.shared_glm_hmm.grouping import session_metadata

In [ ]:
processed_dir = Path(dir_config.data.processed)
shared_dir = processed_dir / "shared_glm_hmm"

CONFIG = "ashwood_color_non_standardized"   # run_name; add "__colorpm1" if fit with --color-coding pm1
final_path = shared_dir / CONFIG / "all_subjects_final.pkl"

bundle = pickle.load(open(final_path, "rb"))
pooled = bundle["model"]["pooled"]
feats = bundle["config"]["model_features"]
K = bundle["best_k"]
data = bundle["data"]

metadata = pd.read_csv(processed_dir / "processed_metadata_all_data_accu_60.csv")
sess_md = session_metadata(metadata).set_index("session_id")
print(f"config={CONFIG}  K={K}  n_sessions={len(data)}  features={feats}")

## 1. Per-state weights & the color-bias state

States are defined once by the shared pooled model. Identify the color-bias state as the
one with the largest `color` weight.

In [ ]:
w = -pooled.observations.params[:, 0, :]   # (K, M)
color_idx = feats.index("color")
colorbias_state = int(np.argmax(w[:, color_idx]))

wdf = pd.DataFrame(w, columns=feats, index=[f"S{k}" for k in range(K)])
print(f"color-bias state = S{colorbias_state}  (color weight = {w[colorbias_state, color_idx]:.3f})")
wdf.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(feats))
for k in range(K):
    lw = 3 if k == colorbias_state else 1.5
    ax.plot(x, w[k], marker="o", lw=lw, label=f"S{k}" + (" (color-bias)" if k == colorbias_state else ""))
ax.axhline(0, color="k", ls="--", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(feats, rotation=20, ha="right")
ax.set(ylabel="GLM weight", title=f"{CONFIG} — shared pooled state weights")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 2. Per-session occupancy under the shared pooled model

Occupancy = mean posterior state probability over a session's valid trials, computed with
the **single shared model** (identical basis for every session — no refit, no alignment).

In [ ]:
def session_occupancy(model, df, feats):
    obs = df["choices"].values.astype(int).reshape(-1, 1)
    inp = df[feats].values.astype(float)
    msk = df["mask"].values.astype(bool).reshape(-1, 1)
    post = model.expected_states(data=obs, input=inp, mask=msk)[0]   # (T, K)
    return post[msk[:, 0]].mean(axis=0)

rows = []
for sid, df in data.items():
    occ = session_occupancy(pooled, df, feats)
    row = {"session_id": sid, **{f"occ_S{k}": occ[k] for k in range(K)}}
    rows.append(row)
occ_df = pd.DataFrame(rows).set_index("session_id")
occ_df["occ_colorbias"] = occ_df[f"occ_S{colorbias_state}"]

occ_df = occ_df.join(sess_md)
occ_df["subtype"] = pd.Categorical(occ_df["subtype"], ["HC", "tremor", "bradykinetic", "intermediate"])
print("occupancy table:", occ_df.shape)
occ_df.groupby(["subtype", "medication"], observed=True)["occ_colorbias"].agg(["mean", "sem", "count"]).round(3)

## 3. Within-subject OFF→ON paired test (color-bias state)

Paired Wilcoxon on the 41 PD subjects, overall and per subtype.

In [ ]:
pd_occ = occ_df[occ_df["is_pd"] == 1].copy()
paired = pd_occ.pivot_table(index=["subject_id", "subtype"], columns="medication",
                            values="occ_colorbias", observed=True).dropna(subset=["off", "on"]).reset_index()
paired["delta"] = paired["on"] - paired["off"]

def paired_test(sub):
    d = paired if sub is None else paired[paired["subtype"] == sub]
    if len(d) < 3:
        return dict(subtype=sub or "all_PD", n=len(d), off=np.nan, on=np.nan, delta=np.nan, p=np.nan)
    stat, p = wilcoxon(d["off"], d["on"])
    return dict(subtype=sub or "all_PD", n=len(d), off=d["off"].mean(), on=d["on"].mean(),
                delta=d["delta"].mean(), p=p)

res = pd.DataFrame([paired_test(s) for s in [None, "tremor", "bradykinetic"]])
print("Paired Wilcoxon: OFF vs ON occupancy of color-bias state")
res.round(4)

## 4. Subtype × medication mixed model

`occ_colorbias ~ medication * subtype` with a per-subject random intercept (tremor vs
brady only; the interaction term tests whether medication's effect differs by subtype).

In [ ]:
mm_df = pd_occ[pd_occ["subtype"].isin(["tremor", "bradykinetic"])].copy()
mm_df["medication"] = pd.Categorical(mm_df["medication"], ["off", "on"])
mm_df["subtype"] = mm_df["subtype"].astype(str)
try:
    model = smf.mixedlm("occ_colorbias ~ C(medication) * C(subtype)", mm_df, groups=mm_df["subject_id"])
    fit = model.fit(reml=False)
    print(fit.summary())
except Exception as e:
    print("mixed model failed:", e)

## 5. Within-subject occupancy plot

Per-subject OFF→ON slope lines by subtype, group means ± SEM, and the HC reference band.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
hc_mean = occ_df[occ_df["subtype"] == "HC"]["occ_colorbias"].mean()
hc_sem = occ_df[occ_df["subtype"] == "HC"]["occ_colorbias"].sem()

for ax, sub in zip(axes, ["tremor", "bradykinetic"]):
    d = paired[paired["subtype"] == sub]
    for _, r in d.iterrows():
        ax.plot([0, 1], [r["off"], r["on"]], "-", color="gray", alpha=0.4, lw=1)
        ax.plot([0, 1], [r["off"], r["on"]], "o", color="gray", alpha=0.5, ms=4)
    m = [d["off"].mean(), d["on"].mean()]
    s = [d["off"].sem(), d["on"].sem()]
    ax.errorbar([0, 1], m, yerr=s, color="tab:red", lw=3, marker="o", ms=10, capsize=6, zorder=5)
    ax.axhspan(hc_mean - hc_sem, hc_mean + hc_sem, color="tab:green", alpha=0.15)
    ax.axhline(hc_mean, color="tab:green", ls="--", lw=1.5, label="HC reference")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["OFF", "ON"], fontsize=13)
    ax.set_xlim(-0.3, 1.3)
    ax.set_title(f"{sub}  (n={len(d)})", fontsize=13)
    ax.legend(fontsize=9)
axes[0].set_ylabel("Color-bias state occupancy", fontsize=12)
fig.suptitle(f"{CONFIG} — within-subject medication effect on color-bias state", fontsize=14)
plt.tight_layout(); plt.show()

## 6. All-state occupancy by subtype × medication

Full state-occupancy profile, to see which states (not just color-bias) shift with
medication / differ across subtypes.

In [ ]:
occ_cols = [f"occ_S{k}" for k in range(K)]
summary = occ_df.groupby(["subtype", "medication"], observed=True)[occ_cols].mean()

groups = [("HC", "none"), ("tremor", "off"), ("tremor", "on"),
          ("bradykinetic", "off"), ("bradykinetic", "on")]
groups = [g for g in groups if g in summary.index]
labels = [f"{s}\n{m}" for s, m in groups]

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(groups))
for k in range(K):
    vals = np.array([summary.loc[g, f"occ_S{k}"] for g in groups])
    lbl = f"S{k}" + (" (color-bias)" if k == colorbias_state else "")
    ax.bar(labels, vals, bottom=bottom, label=lbl)
    bottom += vals
ax.set(ylabel="Mean occupancy", title=f"{CONFIG} — state occupancy by group", ylim=(0, 1))
ax.legend(fontsize=8, ncol=K); plt.tight_layout(); plt.show()